<a href="https://colab.research.google.com/github/swaaminathanm/deep-reinforcement-learning-exercises/blob/main/moving-car-rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt install swig cmake
!apt-get install swig
!pip install stable-baselines3[extra] gymnasium[box2d] huggingface_sb3 huggingface_hub
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay
!pip install imageio[ffmpeg]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 67 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 67 not upgraded.
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubu

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [2]:
import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
import gymnasium as gym

env = gym.make("MountainCar-v0")
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

Observation Space Shape (2,)
Sample observation [-1.0185953  0.0130124]


In [5]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

Action Space Shape 3
Action Space Sample 0


In [16]:
env = make_vec_env('MountainCar-v0', n_envs=16)

model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 2048,
    batch_size = 64,
    n_epochs = 6,
    gamma = 0.99,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

Using cuda device


In [21]:
model.learn(total_timesteps=1000000)
model_name = "MountainCar-v0"
model.save(model_name)

Streaming output truncated to the last 5000 lines.
|    value_loss           | 14.9         |
------------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 116          |
|    ep_rew_mean          | -116         |
| time/                   |              |
|    fps                  | 521          |
|    iterations           | 252          |
|    time_elapsed         | 990          |
|    total_timesteps      | 516096       |
| train/                  |              |
|    approx_kl            | 0.0018465314 |
|    clip_fraction        | 0.013        |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.0708      |
|    explained_variance   | 0.929        |
|    learning_rate        | 0.0003       |
|    loss                 | 10.5         |
|    n_updates            | 1692         |
|    policy_gradient_loss | 0.000127     |
|    value_loss           | 23.3         |
---

In [22]:
import gymnasium as gym
from stable_baselines3 import PPO
import imageio

model_name = "MountainCar-v0"

env = gym.make("MountainCar-v0", render_mode="rgb_array")

model = PPO.load(model_name, env=env)

video_frames = []

state, info = env.reset()
done = False
score = 0

while not done:
    # Use deterministic to see the best strategy
    action, _states = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = env.step(action)

    frame = env.render()
    video_frames.append(frame)

    done = terminated or truncated
    score += reward

env.close()
print(f"Game Finished. Total Score: {score}. Captured {len(video_frames)} frames.")

# 3. SAVE AS VIDEO: Write the array to a physical file in your Colab sidebar
video_path = "mountain_car_playback.mp4"
imageio.mimsave(video_path, video_frames, fps=30)
print(f"Video saved successfully as: {video_path}")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Game Finished. Total Score: -148.0. Captured 148 frames.
Video saved successfully as: mountain_car_playback.mp4


In [23]:
from IPython.display import Video
# This will display the video component inside your notebook
Video(video_path, embed=True, width=600, height=400)